# Setup

In [ ]:
!pip install --upgrade pip
!pip install ipywidgets jupyterlab_widgets
!pip install numpy pandas matplotlib seaborn scikit-learn scipy
!pip install librosa soundfile
!pip install gdown
!pip install requests huggingface-hub ipywidgets jupyterlab_widgets omegaconf einops
!pip install setuptools
# !pip install --upgrade setuptools
# !pip install setuptools==81.0.0 --force-reinstall

!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
# !pip3 install torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu118

!pip install stable-audio-tools --no-deps
!pip install safetensors
!pip install encodec
!pip install descript-audio-codec
!pip install git+https://github.com/facebookresearch/dacvae
!pip install bigvgan

!pip install pytorch-lightning

In [ ]:
# Training data
!gdown 'https://drive.google.com/uc?id=1_Rq7dkY4hHhp9WlN4P7PBtthBp8rn6tA' -O bass.zip
!gdown 'https://drive.google.com/uc?id=1Yc4Ft8RVVhiMTQHQ0cnpZdzDSEQpdQPE' -O drums.zip
!gdown 'https://drive.google.com/uc?id=1GUj6C1ieHOdE3UXurcVOuCNmoa1xX45A' -O guitar.zip
!gdown 'https://drive.google.com/uc?id=1fUuECiEmecCQPKa3ifFVud5XqU4tftHY' -O piano.zip

In [ ]:
# Download test data. This is a small subset of the training data, but will be used for testing the model during development.
!gdown 'https://drive.google.com/uc?id=1Xo-bGORndJhenHvzf3eY5Lt0XCTgElZP' -O dataset.zip

In [ ]:
!mkdir data
!tar -xzf dataset.zip -C data

In [ ]:
!mkdir -p pretrained

# Download HiFiGAN vocoder checkpoint
!wget -O pretrained/hifigan.ckpt https://zenodo.org/record/10643148/files/hifigan-ckpt.ckpt

# Download VAE checkpoint
!wget -O pretrained/vae.ckpt https://zenodo.org/record/10643148/files/vae-ckpt.ckpt

# Pretrained model weights
!wget -O pretrained/original.ckpt https://zenodo.org/records/15123184/files/2024-05-23T09-28-56_3_D_4_stems_slakh_mix_cond_sumch_3e-05_zero_unconditional_checkpoint.pt

In [ ]:
import sys
sys.path.insert(0, "./simultaneous-music-separation-and-generation")
sys.path.insert(0, "./simultaneous-music-separation-and-generation/vendor")

## Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import time
import random
import librosa, librosa.display
from IPython.display import Audio, display, HTML
import sys, json, torch, torch.nn as nn
import soundfile as sf
from abc import ABC, abstractmethod
from pathlib import Path
from IPython.display import Audio, display
from typing import List, Dict, Optional, Tuple
import shutil
from datetime import datetime

from utils.audio import display_waveform, display_stft, display_mel
from utils.latent import encode_audio, decode_latent
from utils.saving import save_encoded, load_encoded
from utils.inference import generate_stems, separate_mixture, load_saved_stems, STEM_NAMES
from utils.metrics import evaluate_separation
from utils.saving import save_sep_originals, load_sep_originals

from modules.msgld.msgld_extractor import MsgLdMelExtractor
from modules.msgld.msgld_vae import MsgLdVAE
from modules.msgld.msgld_hifigan import MsgLdHiFiGAN
from modules.msgld.msgld_diffusion_model import MsgLdDiffusionModel
from modules.msgld.msgld_diffusion_trainer import MsgLdDiffusionTrainer
from modules.msgld.msgld_sampler import MsgLdSampler

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cpu


In [2]:
%matplotlib inline

# Data

In [ ]:
dataset_path = 'data/slakh2100/test'

folder_names = sorted([d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d))])
folder_names

In [ ]:
folder_idx = 4

folder_path = os.path.join(dataset_path, folder_names[folder_idx])

wav_files = sorted([os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.lower().endswith('.wav')])

wav_files

In [ ]:
tracks = {}
mix = None

for wf in wav_files:
    y, sr = librosa.load(wf, sr=None)
    tracks[wf] = (y, sr)

    if mix is None:
        mix = np.zeros_like(y)
    mix += y

tracks['mix'] = (mix, sr)

In [ ]:
for track in tracks.keys():
    print(f"{track}: {len(tracks[track][0])} samples at {tracks[track][1]} Hz, duration {len(tracks[track][0]) / tracks[track][1]:.2f} seconds")
    display(Audio(tracks[track][0], rate=tracks[track][1]))

In [ ]:
for track in tracks.keys():
    display_waveform(tracks[track][0], tracks[track][1], os.path.basename(track))
    # display_frequency_graph(tracks[track][0], tracks[track][1], os.path.basename(track))
    display_stft(tracks[track][0], tracks[track][1], os.path.basename(track))
    display_mel(tracks[track][0], tracks[track][1], os.path.basename(track))

# Pipelines

In [3]:
# Paths
VAE_CKPT = "pretrained/vae.ckpt"
HIFIGAN_CKPT = "pretrained/hifigan.ckpt"
HIFIGAN_CONFIG_PATH = "vendor/hifigan/config_16k_64.json"

# Configs
with open(HIFIGAN_CONFIG_PATH) as f:
    hifigan_config = json.load(f)

# Instantiate 
mel_extractor = MsgLdMelExtractor()
vae = MsgLdVAE(vae_ckpt=VAE_CKPT, device=str(DEVICE)).to(DEVICE).eval()
vocoder = MsgLdHiFiGAN(ckpt_path=HIFIGAN_CKPT, config=hifigan_config, device=str(DEVICE)).to(DEVICE).eval()

print(f"VAE loaded – encoder params: {sum(p.numel() for p in vae.encoder.parameters()):,}")
print(f"HiFi-GAN loaded – params: {sum(p.numel() for p in vocoder.model.parameters()):,}")

Working with z of shape (1, 8, 64, 64) = 32768 dimensions.
Loading VAE weights from pretrained/vae.ckpt
  Loaded 88 params for encoder
  Loaded 112 params for decoder
  Loaded 2 params for quant_conv
  Loaded 2 params for post_quant_conv


/home/cim/Desktop/Workspace/Thesis/.venv/lib/python3.14/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Loading HiFi-GAN weights from pretrained/hifigan.ckpt
Removing weight norm...
VAE loaded – encoder params: 22,395,024
HiFi-GAN loaded – params: 55,264,897


## Check: Round-trip

In [ ]:
# Pick a track from the earlier loaded data
track_idx = 4
start_after_seconds = 15

demo_track = list(tracks.keys())[track_idx]
y_demo, sr_demo = tracks[demo_track]
y_demo = y_demo[start_after_seconds*sr_demo:]

# Full round-trip
enc = encode_audio(y_demo, sr_demo, mel_extractor, vae, target_length=1024)
z = enc["z_sample"]  # (1, C, H, W)
dec = decode_latent(z, vae, vocoder)

print(f"Latent shape: {z.shape}")
print(f"Original audio: {len(y_demo)} samples @ {sr_demo} Hz")
print(f"Reconstructed audio: {dec['audio'].shape}")

# Listen to original
print("Original:")
display(Audio(y_demo, rate=sr_demo))

# Listen to reconstruction
print("Reconstructed:")
display(Audio(dec["audio"].squeeze(), rate=mel_extractor.sampling_rate))

## Check: Save & Reload Encoded Latents

In [ ]:
# Encode
enc = encode_audio(y_demo, sr_demo, mel_extractor, vae)
# Save
save_encoded(enc, "demo_latent.npz")
# Load
loaded = load_encoded("demo_latent.npz")
# Decode
dec = decode_latent(loaded["z_sample"], vae, vocoder)

print("Decoded from saved latent:")
display(Audio(dec["audio"].squeeze(), rate=mel_extractor.sampling_rate))

# Clean up
os.remove("demo_latent.npz")

# Diffusions

In [ ]:
diffusion_model = MsgLdDiffusionModel(
    vae=vae,
    mel_extractor=mel_extractor,
    # UNet architecture
    model_channels=128,
    channel_mult=[1, 2, 3, 5],
    num_res_blocks=2,
    attention_resolutions=[8, 4, 2],
    num_head_channels=32,
    # Diffusion schedule
    num_stems=4,
    z_channels=8,
    timesteps=1000,
    
    linear_start=0.0015,
    linear_end=0.0195,
    parameterization="eps",
    conditioning_key="concat",
    # Conditioning
    unconditional_prob=0.0,  # eval - 0.0 / train - 0.1
    # Scale
    scale_by_std=True,
    latent_t_size=256,
    latent_f_size=16,
    device=str(DEVICE),
)
diffusion_model.to(DEVICE)

trainer_module = MsgLdDiffusionTrainer(
    diffusion_model=diffusion_model,
    learning_rate=3e-5,
    warmup_steps=1000,
    l_simple_weight=1.0,
    original_elbo_weight=0.0,
    loss_type="l2",
    learn_logvar=False,
)

sampler = MsgLdSampler(diffusion_model)

unet_params = sum(p.numel() for p in diffusion_model.unet.parameters())
print(f"UNet params: {unet_params:,}")
print(f"Noise schedule: linear beta in [{diffusion_model.linear_start}, {diffusion_model.linear_end}], T={diffusion_model.num_timesteps}")
print(f"Conditioning: {diffusion_model.conditioning_key}")
print(f"Parameterization: {diffusion_model.parameterization}")
print(f"Stems: {diffusion_model.num_stems}, z_channels: {diffusion_model.z_channels}")
print(f"Latent shape: (B, {diffusion_model.num_stems}, {diffusion_model.z_channels}, {diffusion_model.latent_t_size}, {diffusion_model.latent_f_size})")

In [ ]:
CKPT_PATH = "pretrained/original.ckpt"

if os.path.exists(CKPT_PATH):
    print(f"Loading checkpoint: {CKPT_PATH}")
    ckpt = torch.load(CKPT_PATH, map_location="cpu")
    sd = ckpt.get("state_dict", ckpt)

    # Extract UNet weights
    unet_sd = {
        k.replace("model.diffusion_model.", ""): v
        for k, v in sd.items()
        if k.startswith("model.diffusion_model.")
    }
    
    assert len(unet_sd) != 0, "No UNet weights found in checkpoint!"
    
    missing, unexpected = diffusion_model.unet.load_state_dict(unet_sd, strict=False)
    print(f"UNet: loaded {len(unet_sd)} params, missing={len(missing)}, unexpected={len(unexpected)}")

    # Extract scale_factor if present
    if "scale_factor" in sd:
        diffusion_model.scale_factor = sd["scale_factor"].item() if isinstance(sd["scale_factor"], torch.Tensor) else sd["scale_factor"]
        diffusion_model._scale_computed = True
        print(f"scale_factor = {diffusion_model.scale_factor:.4f}")

    # Extract logvar if present
    if "logvar" in sd:
        diffusion_model.logvar = sd["logvar"]
        print("logvar loaded")

    del ckpt, sd
else:
    print(f"No checkpoint found at {CKPT_PATH}. starting from scratch")

# Inference

## Generation (Unconditional)

Starts from pure Gaussian noise with a zero conditioning tensor.

The model jointly generates all 4 stems from its learned prior.

In [ ]:
generated_path = "inference/generated"

results = generate_stems(
    sampler,
    vae,
    vocoder,
    n_samples=2,
    ddim_steps=2,
    ddim_eta=1.0,
    out_dir=generated_path,
)
print(f"Generated {len(results)} sample(s)")
print(f"Sample rate : {vocoder.sample_rate} Hz")

for sample_idx, (gen_audio, gen_mels) in enumerate(results):
    print(f"Sample {sample_idx + 1}")
    print(f"Audio shape : {gen_audio.shape}  dtype: {gen_audio.dtype}")
    for i, name in enumerate(STEM_NAMES):
        print(f"Generated {name}")
        display(Audio(gen_audio[i], rate=vocoder.sample_rate))

In [ ]:
res_g = load_saved_stems(f"{generated_path}/sample_0000")
res_g

## Separation (Conditioned)

Start from pure noise and then generate 4 stems from the mixture latent via reverse-diffusion.

CFG scale 3.0 is a reasonable starting point; increase for sharper separation at the cost of slightly less audio naturalness.

In [5]:
originals_path = "inference/sep_originals"
separated_path = "inference/separated"
tracks_dir = "data/slakh2100/test"
amount_to_separate = 4
list_size = 2
start_from = 0

os.makedirs(originals_path, exist_ok=True)
os.makedirs(separated_path, exist_ok=True)

In [ ]:
clip_samples = int(mel_extractor.sampling_rate * 10.24)
all_folders = sorted([d for d in os.listdir(tracks_dir) if os.path.isdir(os.path.join(tracks_dir, d))])

for batch_start in range(0, amount_to_separate, list_size):
    batch_folders = all_folders[batch_start:batch_start + list_size]
    mix_clips = []
    original_stems_batch = []

    # Prepare the batch
    for folder in batch_folders:
        folder_path = os.path.join(tracks_dir, folder)
        wav_files = sorted([os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.lower().endswith('.wav')])

        wfs = []
        max_len = 0
        for wf in wav_files:
            y, _ = librosa.load(wf, sr=mel_extractor.sampling_rate)
            wfs.append(y)
            max_len = max(max_len, len(y))

        wfs = [np.pad(y, (0, max_len - len(y))) for y in wfs]
        mix = np.zeros(max_len)
        for y in wfs:
            mix += y

        if len(mix) > clip_samples:
            start = random.randint(0, len(mix) - clip_samples)
            mix_clip = mix[start:start + clip_samples]
            stems_clipped = [y[start:start + clip_samples] for y in wfs]
        else:
            mix_clip = mix
            stems_clipped = wfs

        mix_clips.append(mix_clip)
        original_stems_batch.append(stems_clipped)
        print(f"{folder}: {len(mix_clip)} samples")
        
    # Save the data for later evaluation
    save_sep_originals(
        mix_clips,
        original_stems_batch,
        mel_extractor,
        originals_path,
        start_from=start_from,
    )

    # Separate
    print(f"Batch {batch_start // list_size + 1}: separating {len(mix_clips)} mixture")
    results = separate_mixture(
        mix_clips,
        mel_extractor.sampling_rate,
        sampler,
        vae,
        vocoder,
        mel_extractor,
        ddim_steps=1,
        ddim_eta=1.0,
        cfg_scale=3.0,
        out_dir=separated_path,
        start_from=start_from,
    )
    start_from = start_from + list_size
    print(f"Generated {len(results)} separations")
    mix_sr = mel_extractor.sampling_rate

    # Display audio
    # for sample_idx, (sep_audio, sep_mels) in enumerate(results):
    #     print("Original mixture")
    #     display(Audio(mix_clips[sample_idx], rate=mix_sr))
        
    #     print(f"Separation {sample_idx + 1}")
    #     print(f"Audio shape: {sep_audio.shape}  dtype: {sep_audio.dtype}")
    #     for i, name in enumerate(STEM_NAMES):
    #         print(f"Separated {name}")
    #         display(Audio(sep_audio[i], rate=mel_extractor.sampling_rate))
    #     reconstructed_mix = np.sum(sep_audio, axis=0)
    #     print("Reconstructed mix")
    #     display(Audio(reconstructed_mix, rate=mel_extractor.sampling_rate))
    #     break

In [6]:
orig_list = []
sep_list = []

for sample_idx in range(0, 4):
    orig_mix_mel, orig_stem_mels = load_sep_originals(f"{originals_path}/sample_{sample_idx:04d}")
    res_s = load_saved_stems(f"{separated_path}/sample_{sample_idx:04d}")

    orig_mels = orig_stem_mels
    sep_mels = []
    for stem_name in STEM_NAMES:
        sep_mels.append(res_s[stem_name][1])

    orig_mels = torch.stack([torch.tensor(m).unsqueeze(0) for m in orig_mels], dim=0)
    sep_mels = torch.stack([torch.tensor(m).unsqueeze(0) for m in sep_mels], dim=0)
    
    orig_list.append(orig_mels)
    sep_list.append(sep_mels)

per_stem_mse = evaluate_separation(orig_list, sep_list, log=True)

Stem           Mean MSE        Std    N
bass            14.0669     9.0961    4
drums            5.0704     1.6140    4
guitar          23.9139     9.4418    4
piano           19.2706    12.5165    4
Overall         15.5804    11.4779   16


In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
zip_path = f"separated_stems_{timestamp}.zip"

shutil.make_archive(
    zip_path.replace('.zip', ''),
    'zip',
    separated_path
)

print(f"Zipped at: {zip_path}")